In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import torch
print("CUDA :", torch.cuda.is_available())  # doit afficher True

In [ ]:
!nvidia-smi   # doit montrer une Tesla T4

In [ ]:
!pip install -U langchain-text-splitters
!pip install langchain-huggingface
!pip install langchain_chroma
!pip install torch
!pip install -U "bitsandbytes>=0.46.1" transformers accelerate
!pip install rouge-score
!pip install pandas openpyxl

In [ ]:
import pandas as pd
dataset = pd.read_excel('ohada.xlsx')

In [ ]:
dataset.head()

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=120)

documents = []
for _, row in dataset.iterrows():
    texte = f"title : {row['title']}\ncontent : {row['content']}\ndetails : {row['details']}"
    meta = {"title": row['title'], "content": row['content'], "details": row['details']}
    for morceau in splitter.split_text(texte):
        documents.append(Document(page_content=morceau, metadata=meta))

print(len(documents), "chunks")

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 12},
)

try:
    embedding._client.max_seq_length = 512
except Exception:
    pass

In [ ]:
import torch, gc
gc.collect(); torch.cuda.empty_cache()

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory="./chroma_db"
)

In [ ]:
import torch, gc

for attr in ("_client", "client"):
    m = getattr(embedding, attr, None)
    if m is not None:
        try:
            m.to("cpu")
        except Exception:
            pass
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "mistral-community/Mistral-7B-Instruct-v0.3"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={"": 0},
)
model.generation_config.max_length = None

In [ ]:
from transformers import pipeline
from langchain_huggingface.llms import HuggingFacePipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=generator)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("""Tu es un assistant chargé de répondre à des questions. Utilises les éléments de contexte récupérés ci-dessous pour répondre à la question. Si tu ne connais pas la réponse, dis simplement que tu ne la connais pas. Limites ta réponse à trois phrases maximum et restes concis.
Question: {question}
Context: {context}
Answer: """)

def rag_pipeline(query):
    retrieved_docs = vectorstore.similarity_search(
        query,
        k=2
    )

    context = "\n\n".join(
        doc.page_content[:3000]
        for doc in retrieved_docs
    )

    prompt_texte = prompt_template.format(
        question=query,
        context=context
    )

    messages = [{"role": "user", "content": prompt_texte}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    response = llm.invoke(prompt)

    return response.strip(), retrieved_docs

In [ ]:
from IPython.display import display, HTML
import html as _html

def afficher_reponse(response, reference=None):
    body = _html.escape(str(response)).replace("\n", "<br>")
    ref_html = ""
    if reference:
        ref_html = f'<div class="oh-ref"><span class="oh-ref-ic">§</span>{_html.escape(str(reference))}</div>'

    template = """
    <style>
      .oh-card{
        --bg:#ffffff; --fg:#161a20; --muted:#5b6472;
        --edge:rgba(17,24,39,.08); --panel:rgba(124,58,237,.04);
        --a1:#7c3aed; --a2:#06b6d4;
        position:relative; margin:16px 0; padding:1px;
        border-radius:16px; background:linear-gradient(135deg,var(--a1),var(--a2));
        font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif;
        box-shadow:0 10px 30px -12px rgba(124,58,237,.35);
      }
      @media (prefers-color-scheme: dark){
        .oh-card{ --bg:#15181e; --fg:#e8eaed; --muted:#9aa2ad;
          --edge:rgba(255,255,255,.10); --panel:rgba(124,58,237,.10);
          box-shadow:0 10px 30px -12px rgba(0,0,0,.6); }
      }
      .oh-inner{ background:var(--bg); border-radius:15px; padding:18px 20px; }
      .oh-head{ display:flex; align-items:center; gap:10px; margin-bottom:14px; }
      .oh-dot{ width:9px; height:9px; border-radius:50%;
        background:var(--a2); box-shadow:0 0 0 0 rgba(6,182,212,.6);
        animation:oh-pulse 2s infinite; }
      @keyframes oh-pulse{ 0%{box-shadow:0 0 0 0 rgba(6,182,212,.5)}
        70%{box-shadow:0 0 0 7px rgba(6,182,212,0)} 100%{box-shadow:0 0 0 0 rgba(6,182,212,0)} }
      .oh-title{ font-weight:700; font-size:14px; color:var(--fg); letter-spacing:.2px; }
      .oh-badge{ margin-left:auto; font-size:11px; font-weight:600; color:#fff;
        padding:3px 10px; border-radius:999px;
        background:linear-gradient(135deg,var(--a1),var(--a2)); }
      .oh-body{ color:var(--fg); font-size:15px; line-height:1.65;
        border-left:3px solid transparent;
        border-image:linear-gradient(var(--a1),var(--a2)) 1;
        padding:2px 0 2px 16px; }
      .oh-ref{ margin-top:14px; display:inline-flex; align-items:center; gap:8px;
        font-size:12.5px; color:var(--muted); background:var(--panel);
        border:1px solid var(--edge); padding:6px 12px; border-radius:10px; }
      .oh-ref-ic{ font-weight:800; color:var(--a1); }
    </style>
    <div class="oh-card"><div class="oh-inner">
      <div class="oh-head">
        <span class="oh-dot"></span>
        <span class="oh-title">Assistant juridique OHADA</span>
        <span class="oh-badge">RAG</span>
      </div>
      <div class="oh-body">__BODY__</div>
      __REF__
    </div></div>
    """
    display(HTML(template.replace("__BODY__", body).replace("__REF__", ref_html)))

In [ ]:
import re, unicodedata
from collections import Counter

def _strip(s):
    s = unicodedata.normalize('NFD', str(s))
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn').lower()

ACTES = [
    ("AUPSRVE", "AU_recouvrement",  "Acte Uniforme portant organisation des procedures simplifiees de recouvrement et des voies d'execution",
        [r"procedures?\s+simplifiees?\s+de\s+recouvrement", r"voies?\s+d.execution", r"\bAUPSRVE\b", r"\bAUVE\b"]),
    ("AUDSCGIE","AU_societe",       "Acte Uniforme relatif au droit des societes commerciales et du GIE",
        [r"societes?\s+commerciales?", r"groupement\s+d.interet\s+economique", r"\bAUDSCGIE\b", r"\bAUSCGIE\b", r"\bGIE\b"]),
    ("AUDCG",   "AU_commercial",    "Acte Uniforme relatif au droit commercial general",
        [r"droit\s+commercial\s+general", r"\bAUDCG\b", r"registre\s+du\s+commerce"]),
    ("AUS",     "AU_surete",        "Acte Uniforme portant organisation des suretes",
        [r"\bsuretes?\b", r"\bAUS\b"]),
    ("AUPCAP",  "AU_procedures_collectives", "Acte Uniforme portant organisation des procedures collectives d'apurement du passif",
        [r"procedures?\s+collectives?", r"apurement\s+du\s+passif", r"redressement\s+judiciaire", r"liquidation\s+des\s+biens", r"\bAUPCAP\b"]),
    ("AUA",     "AU_arbitrage",     "Acte Uniforme relatif au droit de l'arbitrage",
        [r"\barbitrage\b", r"tribunal\s+arbitral", r"sentence\s+arbitrale", r"\bAUA\b"]),
    ("AUCTMR",  "AU_transport",     "Acte Uniforme relatif aux contrats de transport de marchandises par route",
        [r"transport\s+de\s+marchandises", r"\bAUCTMR\b"]),
    ("AUDCE",   "AU_comptable",     "Acte Uniforme relatif au droit comptable et a l'information financiere",
        [r"droit\s+comptable", r"\bSYSCOHADA\b", r"information\s+financiere"]),
    ("AUSCOOP", "AU_cooperative",   "Acte Uniforme relatif au droit des societes cooperatives",
        [r"societes?\s+cooperatives?", r"\bAUSCOOP\b"]),
    ("AUM",     "AU_mediation",     "Acte Uniforme relatif a la mediation",
        [r"\bmediation\b", r"\bAUM\b"]),
]
ACRO2SLUG = {a[0]: a[1] for a in ACTES}
ACRO2NAME = {a[0]: a[2] for a in ACTES}

def detecter_acte(texte):
    t = _strip(texte)
    best, best_score = None, 0
    for acro, slug, nom, motifs in ACTES:
        score = sum(len(re.findall(m, t)) for m in motifs)
        if score > best_score:
            best, best_score = (acro, slug, nom), score
    return best if best else (None, None, None)

def detecter_article(texte, acte_nom=None):
    t = str(texte)
    arts = [int(m.group(1)) for m in re.finditer(r"[Aa]rticles?\s+(\d{1,4})", t)]
    if not arts:
        return None
    return str(Counter(arts).most_common(1)[0][0])

def extraire_reference(document):
    m = document.metadata
    blob = f"{m.get('content','')} {m.get('details','')} {m.get('title','')}"
    acronyme, slug, _nom = detecter_acte(blob)
    article = detecter_article(blob)
    return acronyme, slug, article

In [ ]:
question = """
Les délibérations du tribunal arbitral sont-elles publiques ou secrètes ?
"""

reponse, docs = rag_pipeline(question)
_, slug, article = extraire_reference(docs[0])
afficher_reponse(reponse, reference=f"{slug} — Article {article}")

In [ ]:
import os

if os.path.exists("Test.csv"):
    test = pd.read_csv("Test.csv")
else:
    test = pd.DataFrame([{"ID": "Q1", "Question": "Les deliberations du tribunal arbitral sont-elles publiques ou secretes ?"}])

lignes = []
for _, row in test.iterrows():
    reponse, retrieved_docs = rag_pipeline(row["Question"])
    _acro, slug, article = extraire_reference(retrieved_docs[0])
    qid = row["ID"]
    lignes.append({"ID": f"{qid}_Answer",                "Target": reponse})
    lignes.append({"ID": f"{qid}_Document_de_Reference", "Target": slug or ""})
    lignes.append({"ID": f"{qid}_Numero_d_Article",      "Target": article or ""})

soumission = pd.DataFrame(lignes)
soumission.to_csv("submission.csv", index=False)
soumission.head(9)

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)

def normaliser_article(valeur):
    try:
        return str(int(valeur))
    except (TypeError, ValueError):
        return None

def calculer_metrique(reponse, acronyme, numero_article, reponse_reference, acte_attendu, article_attendu):
    rouge1 = scorer.score(reponse_reference, reponse)["rouge1"].fmeasure
    exactitude_acte = float(acronyme == acte_attendu)
    exactitude_article = float(numero_article == normaliser_article(article_attendu))
    return 0.4 * rouge1 + 0.3 * exactitude_acte + 0.3 * exactitude_article

In [ ]:
evaluation_reference = pd.read_csv("Eval.csv")

resultats = []
for _, row in evaluation_reference.iterrows():
    reponse, retrieved_docs = rag_pipeline(row["Question"])
    acronyme, slug, numero_article = extraire_reference(retrieved_docs[0])
    score = calculer_metrique(reponse, acronyme, numero_article, row["Answer"], row["Act_Title"], row["Article_Number"])
    resultats.append({
        "ID": row["ID"],
        "reponse": reponse,
        "acte_predit": acronyme,
        "acte_attendu": row["Act_Title"],
        "article_predit": numero_article,
        "article_attendu": row["Article_Number"],
        "score": score,
    })

evaluation = pd.DataFrame(resultats)
evaluation.to_csv("evaluation.csv", index=False)
print("Score moyen :", round(evaluation["score"].mean(), 4))
evaluation